# Análisis del Mercado Laboral Colombiano 2015–2024

Este notebook examina la evolución del desempleo en Colombia durante la última década, con énfasis en:
- Tendencias trimestrales a nivel nacional
- Comparación entre las principales ciudades
- Impacto de la pandemia COVID-19 y recuperación posterior

**Fuente:** DANE – Gran Encuesta Integrada de Hogares (GEIH)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

# Configuración estética
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
sns.set_palette('tab10')
print('Librerías cargadas correctamente.')

## 1. Carga y preparación de datos

In [ ]:
def cargar_datos_desempleo(ruta: str) -> pd.DataFrame:
    """
    Carga y prepara el dataset de desempleo trimestral colombiano.

    Parameters
    ----------
    ruta : str
        Ruta al archivo CSV con los datos históricos.

    Returns
    -------
    pd.DataFrame
        DataFrame con columna de fecha y tasas de desempleo por ciudad.
    """
    df = pd.read_csv(ruta)
    # Crear fecha aproximada: inicio de cada trimestre
    mes_inicio = {1: '01', 2: '04', 3: '07', 4: '10'}
    df['fecha'] = pd.to_datetime(
        df['año'].astype(str) + '-' + df['trimestre'].map(mes_inicio) + '-01'
    )
    return df.sort_values('fecha').reset_index(drop=True)


ruta_csv = Path('../data/processed/desempleo_colombia_2015_2024.csv')
df = cargar_datos_desempleo(ruta_csv)
print(f'Registros cargados: {len(df)}')
print(f'Período: {df["periodo"].iloc[0]} → {df["periodo"].iloc[-1]}')
df.head()

## 2. Estadísticas descriptivas

In [ ]:
ciudades = ['bogota', 'medellin', 'cali', 'barranquilla',
            'bucaramanga', 'manizales', 'ibague', 'pereira', 'cucuta', 'cartagena']

resumen = df[['tasa_nacional'] + ciudades].describe().T
resumen.columns = ['n', 'media', 'std', 'min', 'p25', 'mediana', 'p75', 'max']
resumen = resumen.drop(columns='n').round(2)
resumen.index = resumen.index.str.replace('_', ' ').str.title()
print('Estadísticas descriptivas — Tasa de desempleo (%)')
resumen

## 3. Tendencia nacional trimestral 2015–2024

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df['fecha'], df['tasa_nacional'], color='#1f77b4', linewidth=2.2, label='Tasa nacional')
ax.fill_between(df['fecha'], df['tasa_nacional'], alpha=0.12, color='#1f77b4')

# Marcar pico pandemia
pico = df.loc[df['tasa_nacional'].idxmax()]
ax.annotate(
    f'Pico pandemia\n{pico["periodo"]}: {pico["tasa_nacional"]:.1f}%',
    xy=(pico['fecha'], pico['tasa_nacional']),
    xytext=(pico['fecha'] + pd.DateOffset(months=6), pico['tasa_nacional'] - 2.5),
    arrowprops=dict(arrowstyle='->', color='#d62728'),
    color='#d62728', fontsize=9.5,
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title('Tasa de desempleo nacional — Colombia (2015–2024)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Trimestre')
ax.set_ylabel('Tasa de desempleo')
ax.legend()
plt.tight_layout()
plt.savefig('../assets/desempleo_nacional_tendencia.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Comparación por ciudad principal

In [ ]:
ciudades_principales = ['bogota', 'medellin', 'cali', 'barranquilla', 'bucaramanga']
colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, ax = plt.subplots(figsize=(14, 6))

for ciudad, color in zip(ciudades_principales, colores):
    ax.plot(df['fecha'], df[ciudad], linewidth=1.8, label=ciudad.capitalize(), color=color)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title('Desempleo por ciudad principal — Colombia (2015–2024)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Trimestre')
ax.set_ylabel('Tasa de desempleo')
ax.legend(loc='upper left', framealpha=0.7)
plt.tight_layout()
plt.savefig('../assets/desempleo_por_ciudad.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Análisis del impacto pandémico y recuperación

In [ ]:
def calcular_variacion_pandemia(df: pd.DataFrame, ciudades: list) -> pd.DataFrame:
    """
    Calcula el impacto de la pandemia comparando T1-2020 con T2-2020
    y la recuperación medida en T4-2021.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con datos de desempleo.
    ciudades : list
        Lista de columnas de ciudades a analizar.

    Returns
    -------
    pd.DataFrame
        Tabla con impacto (pp) y porcentaje de recuperación.
    """
    pre  = df[df['periodo'] == '2020-T1'][['tasa_nacional'] + ciudades].iloc[0]
    pico = df[df['periodo'] == '2020-T2'][['tasa_nacional'] + ciudades].iloc[0]
    rec  = df[df['periodo'] == '2021-T4'][['tasa_nacional'] + ciudades].iloc[0]

    resultado = pd.DataFrame({
        'Pre-pandemia (T1-2020)': pre,
        'Pico pandemia (T2-2020)': pico,
        'Recuperación (T4-2021)': rec,
        'Impacto (pp)': (pico - pre).round(1),
        'Recuperación (%)': ((pico - rec) / (pico - pre) * 100).round(1),
    })
    return resultado


tabla_pandemia = calcular_variacion_pandemia(df, ciudades)
tabla_pandemia.index = tabla_pandemia.index.str.replace('_', ' ').str.title()
print('Impacto pandémico y recuperación — puntos porcentuales')
tabla_pandemia

In [ ]:
# Gráfico de barras: impacto pandémico por ciudad
impacto = tabla_pandemia['Impacto (pp)'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
barras = ax.bar(impacto.index, impacto.values, color='#d62728', edgecolor='white', linewidth=0.6)

for barra in barras:
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.2,
        f'+{barra.get_height():.1f} pp',
        ha='center', va='bottom', fontsize=8.5,
    )

ax.set_title('Incremento del desempleo durante la pandemia (T1 → T2 2020)', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Puntos porcentuales adicionales')
ax.set_xlabel('Ciudad / Nacional')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../assets/impacto_pandemia_desempleo.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Tendencia post-pandemia (2021–2024)

In [ ]:
def resumir_tendencia_anual(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula la tasa de desempleo promedio anual a nivel nacional.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con datos trimestrales.

    Returns
    -------
    pd.DataFrame
        Promedio anual con variación interanual en puntos porcentuales.
    """
    anual = df.groupby('año')['tasa_nacional'].mean().reset_index()
    anual.columns = ['año', 'tasa_promedio']
    anual['variacion_pp'] = anual['tasa_promedio'].diff().round(2)
    return anual


tendencia_anual = resumir_tendencia_anual(df)

fig, ax = plt.subplots(figsize=(11, 5))
colores_barra = ['#d62728' if v > 0 else '#2ca02c'
                 for v in tendencia_anual['variacion_pp'].fillna(0)]

ax.bar(tendencia_anual['año'], tendencia_anual['tasa_promedio'],
       color='#aec7e8', edgecolor='#1f77b4', linewidth=0.8, label='Tasa promedio anual')
ax2 = ax.twinx()
ax2.plot(tendencia_anual['año'], tendencia_anual['variacion_pp'],
         'o--', color='#ff7f0e', linewidth=1.6, label='Variación interanual (pp)')
ax2.axhline(0, color='grey', linewidth=0.7, linestyle=':')

ax.set_title('Desempleo promedio anual y variación interanual — Colombia', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Año')
ax.set_ylabel('Tasa promedio anual (%)')
ax2.set_ylabel('Variación (pp)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('../assets/desempleo_anual_variacion.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nResumen tendencia post-pandemia:')
print(tendencia_anual[tendencia_anual['año'] >= 2021].to_string(index=False))

## 7. Conclusiones

### Hallazgos principales

1. **Tendencia pre-pandemia (2015–2019):** La tasa de desempleo nacional se mantuvo en un rango estable de 8.9%–12.6% con marcada estacionalidad: los primeros trimestres presentan siempre el mayor desempleo por ser época de menor actividad económica.

2. **Choque pandémico (T2-2020):** El desempleo nacional saltó de 12.6% (T1-2020) a 21.4% (T2-2020), un incremento de 8.8 puntos porcentuales en un solo trimestre. Las ciudades más afectadas fueron Ibagué (+8.8 pp) y Cúcuta (+9.0 pp), que ya presentaban tasas estructuralmente altas.

3. **Recuperación 2021–2022:** El mercado laboral mostró una recuperación notable: para T4-2021 la tasa nacional había bajado a 11.3%, recuperando aproximadamente el 76% del impacto pandémico. Sin embargo, esta recuperación fue heterogénea por ciudad.

4. **Tendencia 2023–2024:** El desempleo continúa descendiendo aunque a un ritmo más lento, convergiendo hacia los niveles pre-pandemia. Al cierre de 2024 (T4) la tasa nacional alcanzó 9.4%, ligeramente por encima del mínimo histórico reciente (8.9% en T4-2015).

5. **Brecha estructural ciudad–ciudad:** Ibagué consistentemente presenta las tasas más altas (14–27%), mientras Barranquilla y Bucaramanga lideran como las plazas de menor desempleo, reflejando diferencias en la estructura productiva regional.

### Limitaciones
- Los datos corresponden a promedios trimestrales; la variabilidad mensual intra-trimestral puede ser significativa.
- Las cifras de Cúcuta deben interpretarse con cautela dada la alta informalidad transfronteriza con Venezuela.
